In [ ]:
# Import required libraries
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display

# Add src to path
sys.path.append('../src')

from audio_processing.pitch_detect import PitchDetector
from utils.helpers import format_time, print_banner

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

## 1. Load Sample Audio

In [ ]:
# Select an audio file
DATA_DIR = '../data/raw'
ragas = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]

if ragas:
    sample_raga = ragas[0]
    raga_path = os.path.join(DATA_DIR, sample_raga)
    audio_files = [f for f in os.listdir(raga_path) if f.endswith(('.mp3', '.wav', '.m4a'))]
    
    if audio_files:
        AUDIO_FILE = os.path.join(raga_path, audio_files[0])
        print(f"Selected: {AUDIO_FILE}")
    else:
        AUDIO_FILE = None
        print("No audio files found!")
else:
    AUDIO_FILE = None
    print("No raga folders found!")

In [ ]:
# Load audio
if AUDIO_FILE:
    y, sr = librosa.load(AUDIO_FILE, duration=30)
    print(f"Sample rate: {sr} Hz")
    print(f"Duration: {len(y)/sr:.2f} seconds")
    print(f"Samples: {len(y):,}")

## 2. Pitch Detection

In [ ]:
# Initialize pitch detector
detector = PitchDetector(sample_rate=sr, fmin=80, fmax=600)

# Detect pitch using pYIN (more robust)
print("Detecting pitch...")
pitch_contour = detector.detect_pitch_yin(y)

print(f"Pitch contour shape: {pitch_contour.shape}")
print(f"Valid pitches: {np.sum(~np.isnan(pitch_contour))}/{len(pitch_contour)}")

In [ ]:
# Visualize pitch contour
frames = range(len(pitch_contour))
times = librosa.frames_to_time(frames, sr=sr, hop_length=512)

plt.figure(figsize=(14, 4))
plt.plot(times, pitch_contour)
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.title('Raw Pitch Contour')
plt.grid(True)
plt.tight_layout()
plt.show()

## 3. Smooth Pitch Contour

In [ ]:
# Smooth pitch
smoothed_pitch = detector.smooth_pitch(pitch_contour, kernel_size=5)

# Plot comparison
plt.figure(figsize=(14, 4))
plt.plot(times, pitch_contour, alpha=0.3, label='Raw')
plt.plot(times, smoothed_pitch, label='Smoothed', linewidth=2)
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.title('Pitch Smoothing')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 4. Estimate Tonic (Sa)

In [ ]:
# Estimate tonic using histogram method
try:
    tonic = detector.estimate_tonic(smoothed_pitch, method='histogram')
    print(f"Estimated Tonic (Sa): {tonic:.2f} Hz")
    
    # Convert to note name
    from utils.helpers import hz_to_note_name
    note_name = hz_to_note_name(tonic)
    print(f"Tonic Note: {note_name}")
except Exception as e:
    print(f"Error estimating tonic: {e}")

In [ ]:
# Visualize pitch distribution
valid_pitches = smoothed_pitch[~np.isnan(smoothed_pitch)]

plt.figure(figsize=(10, 5))
plt.hist(valid_pitches, bins=50, edgecolor='black', alpha=0.7)
plt.axvline(tonic, color='red', linestyle='--', linewidth=2, label=f'Tonic: {tonic:.2f} Hz')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Count')
plt.title('Pitch Distribution')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Convert to Cents and Notes

In [ ]:
# Convert pitch to cents relative to tonic
cents_contour = np.array([detector.pitch_to_cents(p, tonic) for p in smoothed_pitch])

# Plot in cents
plt.figure(figsize=(14, 4))
plt.plot(times, cents_contour)
plt.xlabel('Time (s)')
plt.ylabel('Cents (relative to Sa)')
plt.title('Pitch Contour in Cents')
plt.axhline(0, color='red', linestyle='--', alpha=0.5, label='Sa (Tonic)')
plt.axhline(1200, color='red', linestyle='--', alpha=0.5, label='Sa (Octave)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Extract note sequence
notes = []
for cents in cents_contour:
    if not np.isnan(cents):
        note, deviation = detector.cents_to_note(cents)
        notes.append(note)

# Get unique notes
unique_notes = list(set([n for n in notes if n]))
print(f"\nUnique notes detected: {sorted(unique_notes)}")

# Count note frequencies
from collections import Counter
note_counts = Counter(notes)
print("\nNote frequencies:")
for note, count in note_counts.most_common(10):
    print(f"  {note}: {count}")

## 6. Visualize Notes Over Time

In [ ]:
# Map notes to numeric values for plotting
note_map = {'S': 0, 'R1': 1, 'R2': 2, 'G2': 3, 'G3': 4, 
            'M1': 5, 'M2': 6, 'P': 7, 'D1': 8, 'D2': 9, 'N2': 10, 'N3': 11}

note_values = [note_map.get(n, np.nan) for n in notes]

plt.figure(figsize=(14, 4))
plt.plot(times[:len(note_values)], note_values, marker='.', linestyle='-', markersize=2)
plt.xlabel('Time (s)')
plt.ylabel('Note')
plt.yticks(range(12), list(note_map.keys()))
plt.title('Note Sequence Over Time')
plt.grid(True, axis='y')
plt.tight_layout()
plt.show()

## 7. Save Results

In [ ]:
# Save pitch analysis results
import json

results = {
    'audio_file': os.path.basename(AUDIO_FILE),
    'tonic_hz': float(tonic),
    'tonic_note': note_name,
    'unique_notes': unique_notes,
    'note_counts': dict(note_counts.most_common())
}

output_file = '../data/notes/pitch_analysis_results.json'
os.makedirs(os.path.dirname(output_file), exist_ok=True)

with open(output_file, 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to: {output_file}")

## Conclusion

This notebook demonstrated:
- Pitch detection using pYIN algorithm
- Pitch smoothing techniques
- Tonic identification
- Conversion from Hz to cents and note names
- Visualization of pitch contours

**Next Steps:**
- Proceed to `03_note_extraction.ipynb` for arohanam/avarohanam extraction